# 04 — LangChain agent integration *(optional, needs API key)*

Notebooks 01–03 talked to STM using the raw MCP client so
you could see every wire-level call. That's great for
understanding how STM behaves, but it's not how you would
*use* STM day-to-day. In practice you put a real agent
framework in front of it.

This notebook shows the simplest real-world integration:
**LangChain `create_agent` + `langchain-mcp-adapters`**.
`create_agent` is LangChain v1's canonical agent factory
and returns a LangGraph graph under the hood, so using it
also exercises LangGraph.

**You will learn:**

- How `langchain-mcp-adapters.load_mcp_tools` converts
  STM's proxied tools into LangChain `BaseTool` instances
- How to build a ReAct-style agent with `create_agent` and
  run it end-to-end
- How to verify STM saw every tool call by reading its
  stats after the agent finishes

**Prereqs:**

- `uv sync --extra langchain` — installs
  `langchain`, `langchain-mcp-adapters`, and
  `langchain-anthropic`
- `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` set in your
  environment (the notebook auto-detects and skips if
  neither is present)

**CI behavior:** This notebook is **excluded from CI** — no
API keys in CI. Run it manually when you want to see the
full agent loop.

## 1. Detect API key

This notebook needs an LLM API key (Anthropic or OpenAI) to
actually run the agent. The cell below sets a ``_HAS_KEY``
flag; every downstream cell short-circuits if the key is
missing so the notebook still renders cleanly without one.
CI does not run this notebook at all.

In [ ]:
import os

_has_anthropic = bool(os.environ.get("ANTHROPIC_API_KEY"))
_has_openai = bool(os.environ.get("OPENAI_API_KEY"))
_HAS_KEY = _has_anthropic or _has_openai

if not _HAS_KEY:
    print(
        "⚠️  Neither ANTHROPIC_API_KEY nor OPENAI_API_KEY is set.\n"
        "   This notebook will render its cells but skip the\n"
        "   actual agent invocation. Set a key and re-run to\n"
        "   execute the full flow:\n"
        "     export ANTHROPIC_API_KEY=sk-ant-...\n"
        "     export OPENAI_API_KEY=sk-..."
    )
else:
    print(f"Anthropic key: {'yes' if _has_anthropic else 'no'}")
    print(f"OpenAI key:    {'yes' if _has_openai else 'no'}")

## 2. Isolate state and bootstrap helpers

In [ ]:
# Add notebooks/ to sys.path so `_helpers` is importable regardless of
# where Jupyter was launched (from the repo root or from notebooks/).
import sys
from pathlib import Path

_cwd = Path.cwd()
if (_cwd / "_helpers.py").exists():
    sys.path.insert(0, str(_cwd))
elif (_cwd / "notebooks" / "_helpers.py").exists():
    sys.path.insert(0, str(_cwd / "notebooks"))
else:
    raise RuntimeError(
        "Cannot find notebooks/_helpers.py — run Jupyter from the repo "
        "root or from the notebooks/ directory."
    )

In [ ]:
import subprocess
import tempfile
from pathlib import Path

from _helpers import (
    isolate_stm_state,
    stm_session,
    extract_text,
    fixtures_dir,
)

config_path = isolate_stm_state(prefix="nb04_")

# Create a tiny sandbox directory with a couple of files
# for the agent to list and read.
sandbox = Path(tempfile.mkdtemp(prefix="nb04_sandbox_"))
(sandbox / "notes.md").write_text(
    "# Notes\n\nmemtomem-stm compresses noisy MCP responses.\n"
    "It also injects relevant long-term memories proactively.\n"
)
(sandbox / "TODO.txt").write_text("- Write more notebooks\n- Ship v0.2\n")
print(f"Sandbox: {sandbox}")
print(f"Files:   {sorted(p.name for p in sandbox.iterdir())}")

## 3. Register the filesystem MCP server as a docfix fallback

For full reproducibility without external NPM packages, we
reuse the `docfix` fixture from notebook 02 instead of the
official `@modelcontextprotocol/server-filesystem`. The
agent will ask about it using natural language and STM will
compress the long response transparently.

In [ ]:
doc_script = fixtures_dir() / "doc_mcp.py"
subprocess.run(
    [
        "uv", "run", "mms", "add", "docfix",
        "--config", str(config_path),
        "--command", "uv",
        "--args", f"run python {doc_script}",
        "--prefix", "docfix",
        "--compression", "selective",
        "--max-chars", "1500",
    ],
    check=True, capture_output=True,
)
print("Registered docfix (selective compression)")

## 4. Load STM's tools into a LangChain agent

The key bridge here is `load_mcp_tools(session)` from
`langchain-mcp-adapters`. It introspects the MCP session,
reads every tool's JSON schema, and wraps each one as a
LangChain `BaseTool` that `create_agent` can consume.

In [ ]:
result = None
stats_text = "(skipped — no API key)"

if _HAS_KEY:
    from mcp import ClientSession
    from mcp.client.stdio import StdioServerParameters, stdio_client
    from langchain_mcp_adapters.tools import load_mcp_tools
    from langchain.agents import create_agent

    model_id = "anthropic:claude-sonnet-4-5" if _has_anthropic else "openai:gpt-4.1-mini"
    print(f"Using model: {model_id}")

    # stdio_client / ClientSession must stay open for the lifetime
    # of agent.ainvoke — LangChain tool calls reach back into the
    # live MCP session. So we do everything inside one async block.
    params = StdioServerParameters(
        command="memtomem-stm", args=[], env=dict(os.environ),
    )
    import os as _os
    with open(_os.devnull, "w") as _devnull:
        async with stdio_client(params, errlog=_devnull) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await load_mcp_tools(session)
                print(f"Loaded {len(tools)} tools from STM into LangChain")
                for t in tools[:8]:
                    print(f"  - {t.name}")
                if len(tools) > 8:
                    print(f"  ... and {len(tools) - 8} more")

                agent = create_agent(model_id, tools)

                result = await agent.ainvoke({
                    "messages": [
                        {
                            "role": "user",
                            "content": (
                                "Use the docfix__get_document tool to list what "
                                "sections are available, then use stm_proxy_select_chunks "
                                "to retrieve just the 'Selective' and 'Surfacing' sections "
                                "and summarize them in 2 sentences."
                            ),
                        }
                    ]
                })

                # Capture stats before the session closes
                stats_raw = await session.call_tool("stm_proxy_stats", {})
                stats_text = extract_text(stats_raw)
else:
    print("(skipped — set ANTHROPIC_API_KEY or OPENAI_API_KEY to run)")

## 5. What the agent did

In [ ]:
if result is None:
    print("(no result — notebook ran without an API key)")
else:
    # result["messages"] is the full conversation including tool calls
    for msg in result["messages"]:
        role = getattr(msg, "type", type(msg).__name__)
        content = getattr(msg, "content", "")
        if isinstance(content, list):
            content = " ".join(
                part.get("text", str(part)) if isinstance(part, dict) else str(part)
                for part in content
            )
        snippet = (str(content)[:240] + "...") if len(str(content)) > 240 else str(content)
        print(f"[{role}] {snippet}")
        print()

## 6. And what STM saw

In [ ]:
print(stats_text)

## Recap

You just ran a LangChain ReAct agent whose entire tool
surface came from STM. Every call the agent made — reading
the document, selecting chunks, checking stats — flowed
through STM's pipeline: cleaned, compressed, cached,
counted. The agent code didn't know STM exists.

**What to try next:**

- Replace the `docfix` fixture with a real upstream like
  `@modelcontextprotocol/server-filesystem` or
  `@modelcontextprotocol/server-github`
- Add a second `mms add` call for a second upstream and
  watch `load_mcp_tools` discover both
- Swap `create_agent` for a custom `StateGraph` from
  LangGraph if you need more control over the loop
- Enable surfacing (see notebook 03) and watch memories
  appear in the agent's observations automatically

## Where to next

- [LangChain `create_agent` docs](https://docs.langchain.com/oss/python/langchain/agents)
- [`langchain-mcp-adapters` on PyPI](https://pypi.org/project/langchain-mcp-adapters/)
- [`docs/configuration.md`](../docs/configuration.md) — env
  vars and config file reference for tuning STM